# 07 · Optimización Delta — costo y rendimiento a escala

En este lab: **Liquid Clustering**, `OPTIMIZE` / compaction, **file sizing**, compresión **ZSTD**,
y cómo medir todo con `DESCRIBE DETAIL`.

> Requiere el lab **01** (usa tu tabla `delta_obras`).

In [0]:
%run ./_setup

## 1. Punto de partida: ¿cómo está tu tabla?
`DESCRIBE DETAIL` muestra número de archivos y tamaño. Muchos archivos pequeños = lecturas lentas.

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {tbl('delta_obras')}")
        .select("numFiles", "sizeInBytes", "clusteringColumns"))

## 2. Liquid Clustering
Reemplaza al particionado rígido y al `ZORDER`. Agrupa físicamente los datos por las columnas que
más se filtran → menos archivos leídos (**data skipping**).

Para CFE las consultas típicas filtran por `IdProyecto`, `IdTipoObra`, `kV`.

> ⚠️ El clustering necesita **estadísticas** en esas columnas. Deben estar entre las
> primeras 32 columnas, o subir `delta.dataSkippingNumIndexedCols`.

In [0]:
spark.sql(f"ALTER TABLE {tbl('delta_obras')} CLUSTER BY (IdProyecto, IdTipoObra, kV)")
spark.sql(f"OPTIMIZE {tbl('delta_obras')}")
display(spark.sql(f"DESCRIBE DETAIL {tbl('delta_obras')}").select("numFiles", "sizeInBytes", "clusteringColumns"))

## 3. Data skipping en acción
Con clustering, una consulta filtrada lee menos archivos.

In [0]:
spark.sql(f"SELECT COUNT(*) FROM {tbl('delta_obras')} WHERE IdProyecto = 3").show()
print(spark.sql(f"EXPLAIN FORMATTED SELECT COUNT(*) FROM {tbl('delta_obras')} WHERE IdProyecto = 3")
      .collect()[0][0][:1500])

## 4. Compresión ZSTD y file sizing
**ZSTD** es el códec recomendado (mejor ratio y velocidad que Snappy/GZIP).

In [0]:
spark.sql(f"""
  ALTER TABLE {tbl('delta_obras')} SET TBLPROPERTIES (
    'delta.parquet.compression.codec' = 'zstd',
    'delta.targetFileSize' = '134217728'
  )
""")
spark.sql(f"OPTIMIZE {tbl('delta_obras')}")
display(spark.sql(f"DESCRIBE DETAIL {tbl('delta_obras')}").select("numFiles", "sizeInBytes"))

## 5. Mantenimiento: VACUUM y compaction automática
* **Compaction automática:** `delta.autoOptimize.optimizeWrite=true` compacta al escribir.
* **VACUUM:** borra archivos viejos no referenciados (libera storage).

```sql
ALTER TABLE <tabla> SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true');
VACUUM <tabla> RETAIN 168 HOURS;   -- 7 días (default)
```